In [2]:
from langchain_community.chat_message_histories import ChatMessageHistory

# Task 1: Ek global dictionary bana rahe hain jo sabhi users ka data store karegi
# Production mein ye dictionary ki jagah Redis ya database (jaise Postgres/MongoDB) use hota hai
store = {}

# Task 2: Router function - Ye decide karega kis user ko kaunsi history deni hai
def get_session_history(session_id: str) -> ChatMessageHistory:
    # Agar user pehli baar aaya hai, toh uske liye nayi Khata Book banao
    if session_id not in store:
        print(f"[SYSTEM] Nayi Khata Book ban rahi hai user ke liye: {session_id}")
        store[session_id] = ChatMessageHistory()
    
    # Agar user ka data pehle se hai, toh wahi purani Khata Book return karo
    return store[session_id]

print("--- Multi-Tenant System Initialized ---\n")

# Task 3: User_Karthik aur User_Guru ke liye alag-alag messages daalna

# Karthik aaya
print("Routing for Karthik...")
karthik_history = get_session_history("User_Karthik")
karthik_history.add_user_message("Hi, I am Karthik. I am an automation engineer.")

# Guru aaya (uska data alag bucket mein jayega)
print("\nRouting for Guru...")
guru_history = get_session_history("User_Guru")
guru_history.add_user_message("Hey, I am Guru. I want to learn LangChain.")

# Karthik wapas aaya (Testing memory persistence)
print("\nKarthik wapas aaya, purani khata book check kar rahe hain...")
karthik_history_again = get_session_history("User_Karthik")
karthik_history_again.add_ai_message("Hello Karthik! Automation is a great field.")

print("\n" + "="*40 + "\n")

# 4. Definition of Done (Verification)
print("--- Final Verdict: Checking Isolation ---")

print("\n👤 User_Karthik ki History:")
for msg in store["User_Karthik"].messages:
    print(f"[{type(msg).__name__}] {msg.content}")

print("\n👤 User_Guru ki History:")
for msg in store["User_Guru"].messages:
    print(f"[{type(msg).__name__}] {msg.content}")

Khata Book (Memory) mein entries daal rahe hain...

--- Checking the Khata Book Structure ---
Role/Type: HumanMessage | Content: Hi, I am an automation engineer.
Role/Type: AIMessage | Content: Hello! Nice to meet you. How can I help you with automation today?
Role/Type: HumanMessage | Content: What is my profession?

--- Raw List Output (Definition of Done) ---
[HumanMessage(content='Hi, I am an automation engineer.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello! Nice to meet you. How can I help you with automation today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my profession?', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
# 👇 Ye naya import hai! Dictionary ki jagah SQL wali class.
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

print("Loading local model...")
# Sahi aur stable connection tera local Ollama server ke saath
llm = ChatOllama(
    base_url="http://localhost:11434", # <-- Ye sabse zaroori tha!
    model="llama3.1:8b",
    temperature=0.2,
    max_tokens=512
)

# 1. Prompt and Chain Setup (Same as Level 4)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant with a permanent memory."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{question}")
])

basic_chain = prompt | llm | StrOutputParser()

# 2. THE UPGRADED RECEPTIONIST (SQL Router)
def get_session_history(session_id: str):
    # Ab dictionary check karne ki zaroorat nahi. 
    # Ye directly SQLite database file se connect karega.
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///meri_khata_book.db" # <-- Yahan badlaav karna hai
    )

# 3. Manager (Wrapper) Setup
chain_with_history = RunnableWithMessageHistory(
    basic_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

# --- LET'S TEST DISK PERSISTENCE ---
config_karthik = {"configurable": {"session_id": "User_Karthik"}}

print("\n--- Call 1: Telling it something new ---")
print("User: My favorite car is a Mustang.")
response_1 = chain_with_history.invoke(
    {"question": "My favorite car is a Mustang."},
    config=config_karthik
)
print(f"AI: {response_1.strip()}")

print("\n--- Call 2: Testing Memory ---")
print("User: Which car do I like?")
response_2 = chain_with_history.invoke(
    {"question": "Which car do I like?"},
    config=config_karthik
)
print(f"AI: {response_2.strip()}")

print("\n[SUCCESS] Script run ho gayi! Ab apne code wale folder mein dekho, 'meri_khata_book.db' naam ki ek file ban gayi hogi.")

Loading local model...

--- Call 1: Telling it something new ---
User: My favorite car is a Mustang.


c:\Users\satyam\Desktop\experiment\.venv\Lib\site-packages\langchain_core\runnables\history.py:596: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(


AI: I remember! You said your favorite car is the Ford Mustang. We were just discussing which generation or model year of Mustang you prefer, but we didn't quite get there yet. Would you like to continue exploring different models and eras of the Mustang?

--- Call 2: Testing Memory ---
User: Which car do I like?
AI: You like a Mustang! Specifically, your favorite car is a Ford Mustang.

[SUCCESS] Script run ho gayi! Ab apne code wale folder mein dekho, 'meri_khata_book.db' naam ki ek file ban gayi hogi.
